In [115]:
import os, sys
PROJECT_ROOT = os.path.abspath("..")  # 从 notebook/ 回到项目根目录
sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "IEEE")
LOG_DIR  = os.path.join(PROJECT_ROOT, "log")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("LOG_DIR:", LOG_DIR)

PROJECT_ROOT: /root/ADHD-EEG-ViT-myversion
DATA_DIR: /root/ADHD-EEG-ViT-myversion/IEEE
LOG_DIR: /root/ADHD-EEG-ViT-myversion/log


In [116]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

from utils import (
    ignore_warnings,
    fix_random_seed,
    device,
    clear_cache,
    join_drive_path,
    log_json,
    train_with_kfold,
    WarmupScheduler,
    evaluate,
    Config,
    IEEEDataConfig,
    EEGDataset,
)
from models.transformer import TransformerConfig, ViTransformer

ignore_warnings()
fix_random_seed(42)
#device = device(force_cuda=True)
#print("Device:", device)
device = device(force_cuda=torch.cuda.is_available())
print("Device:", device)

Device: cuda


In [117]:
config = Config(
    name="ieee transformer",
    batch=8,
    epochs=50,
    lr=1e-3,
    enable_fp16=True,
    grad_step=4,
    warmup_steps=30,
    lr_decay_factor=0.5,
    weight_decay=1e-3,
    patience=30,
)
config.add(k_folds=5)
data_config = IEEEDataConfig()
model_config = TransformerConfig(
    embed_dim=64,
    num_heads=4,
    num_blocks=4,
    block_hidden_dim=128,
    fc_hidden_dim=32,
    dropout=0.1,
)

print("ID:", config.id)
print("Name:", config.name)

ID: 260309004348772585
Name: ieee-transformer


In [118]:
train_data_path = os.path.join(DATA_DIR, data_config.train)  # train.pt
val_data_path   = os.path.join(DATA_DIR, data_config.val)    # val.pt

train_data = torch.load(train_data_path, weights_only=True)
val_data = torch.load(val_data_path, weights_only=True)

# Concat Train-set and Validation-set for Cross validation
signals = torch.cat([train_data["data"], val_data["data"]], dim=0)
labels = torch.cat([train_data["label"], val_data["label"]], dim=0)

train_dataset = EEGDataset({"data": signals, "label": labels})

In [119]:
model_param = {
    "input_channel": data_config.channels,
    "seq_length": data_config.length,
    "embed_dim": model_config.embed_dim,
    "num_heads": model_config.num_heads,
    "num_blocks": model_config.num_blocks,
    "block_hidden_dim": model_config.block_hidden_dim,
    "fc_hidden_dim": model_config.fc_hidden_dim,
    "num_classes": data_config.num_classes,
    "dropout_p": model_config.dropout,
    
    # "use_channel_attn": True,           #3.5channelattention动态态
    # "channel_attn_type": "dynamic",
    # "channel_attn_reduction": 4,

    # "use_channel_attn": True,           #3.5channelattention静态
    # "channel_attn_type": "static",
}

criterion = nn.CrossEntropyLoss()
check_point, best_model_path = train_with_kfold(
    k_folds=config.k_folds,
    model_class=ViTransformer,
    device=device,
    model_path=config.model_path,
    optimizer_class=optim.Adam,
    criterion=criterion,
    epochs=config.epochs,
    train_dataset=train_dataset,
    batch=config.batch,
    gradient_step=config.grad_step,
    patience=config.patience,
    model_params=model_param,
    optimizer_params={"lr": config.lr, "weight_decay": config.weight_decay},
    enable_fp16=config.enable_fp16,
    scheduler_class=WarmupScheduler,
    scheduler_params={
        "lr": config.lr,
        "warmup_steps": config.warmup_steps,
        "decay_factor": config.lr_decay_factor,
    },
)


===== Fold 1 =====


  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1, Train-Loss: 0.84230,  Val-Loss: 0.88923
Epoch 2, Train-Loss: 0.81539,  Val-Loss: 0.83471
Epoch 3, Train-Loss: 0.75772,  Val-Loss: 0.76770
Epoch 4, Train-Loss: 0.71558,  Val-Loss: 0.69862
Epoch 5, Train-Loss: 0.68883,  Val-Loss: 0.64464
Epoch 6, Train-Loss: 0.71456,  Val-Loss: 0.61665
Epoch 7, Train-Loss: 0.69182,  Val-Loss: 0.60590
Epoch 8, Train-Loss: 0.69108,  Val-Loss: 0.60294
Epoch 9, Train-Loss: 0.66322,  Val-Loss: 0.60697
Epoch 10, Train-Loss: 0.67686,  Val-Loss: 0.62222
Epoch 11, Train-Loss: 0.63486,  Val-Loss: 0.64250
Epoch 12, Train-Loss: 0.64365,  Val-Loss: 0.66129
Epoch 13, Train-Loss: 0.63813,  Val-Loss: 0.66158
Epoch 14, Train-Loss: 0.62530,  Val-Loss: 0.64232
Epoch 15, Train-Loss: 0.59791,  Val-Loss: 0.62878
Epoch 16, Train-Loss: 0.59695,  Val-Loss: 0.62021
Epoch 17, Train-Loss: 0.60335,  Val-Loss: 0.61999
Epoch 18, Train-Loss: 0.54596,  Val-Loss: 0.63008
Epoch 19, Train-Loss: 0.55627,  Val-Loss: 0.61848
Epoch 20, Train-Loss: 0.53653,  Val-Loss: 0.63539
Epoch 21,

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1, Train-Loss: 1.51854,  Val-Loss: 1.28157
Epoch 2, Train-Loss: 1.48688,  Val-Loss: 1.18353
Epoch 3, Train-Loss: 1.27892,  Val-Loss: 1.05539
Epoch 4, Train-Loss: 1.09315,  Val-Loss: 0.91091
Epoch 5, Train-Loss: 0.95397,  Val-Loss: 0.79266
Epoch 6, Train-Loss: 0.81670,  Val-Loss: 0.72522
Epoch 7, Train-Loss: 0.72593,  Val-Loss: 0.70917
Epoch 8, Train-Loss: 0.69036,  Val-Loss: 0.73944
Epoch 9, Train-Loss: 0.66992,  Val-Loss: 0.78359
Epoch 10, Train-Loss: 0.68417,  Val-Loss: 0.79778
Epoch 11, Train-Loss: 0.69287,  Val-Loss: 0.78377
Epoch 12, Train-Loss: 0.69737,  Val-Loss: 0.75043
Epoch 13, Train-Loss: 0.70416,  Val-Loss: 0.72316
Epoch 14, Train-Loss: 0.67972,  Val-Loss: 0.70713
Epoch 15, Train-Loss: 0.63041,  Val-Loss: 0.69556
Epoch 16, Train-Loss: 0.64356,  Val-Loss: 0.69048
Epoch 17, Train-Loss: 0.65062,  Val-Loss: 0.68652
Epoch 18, Train-Loss: 0.63177,  Val-Loss: 0.68611
Epoch 19, Train-Loss: 0.59695,  Val-Loss: 0.68670
Epoch 20, Train-Loss: 0.62760,  Val-Loss: 0.68980
Epoch 21,

...

Epoch 1, Train-Loss: 0.75106,  Val-Loss: 0.67358
Epoch 2, Train-Loss: 0.68137,  Val-Loss: 0.67134
Epoch 3, Train-Loss: 0.69996,  Val-Loss: 0.67051
Epoch 4, Train-Loss: 0.70157,  Val-Loss: 0.66803
Epoch 5, Train-Loss: 0.69038,  Val-Loss: 0.66641
Epoch 6, Train-Loss: 0.68752,  Val-Loss: 0.66415
Epoch 7, Train-Loss: 0.62131,  Val-Loss: 0.66094
Epoch 8, Train-Loss: 0.66668,  Val-Loss: 0.65768
Epoch 9, Train-Loss: 0.59403,  Val-Loss: 0.65498
Epoch 10, Train-Loss: 0.59998,  Val-Loss: 0.65306
Epoch 11, Train-Loss: 0.56255,  Val-Loss: 0.65206
Epoch 12, Train-Loss: 0.55914,  Val-Loss: 0.64497
Epoch 13, Train-Loss: 0.55183,  Val-Loss: 0.64225
Epoch 14, Train-Loss: 0.49828,  Val-Loss: 0.65062
Epoch 15, Train-Loss: 0.45711,  Val-Loss: 0.62969
Epoch 16, Train-Loss: 0.51611,  Val-Loss: 0.62609
Epoch 17, Train-Loss: 0.47320,  Val-Loss: 0.63366
Epoch 18, Train-Loss: 0.41327,  Val-Loss: 0.62020
Epoch 19, Train-Loss: 0.39088,  Val-Loss: 0.61811
Epoch 20, Train-Loss: 0.35950,  Val-Loss: 0.61122
Epoch 21,

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1, Train-Loss: 1.36540,  Val-Loss: 1.21111
Epoch 2, Train-Loss: 1.27672,  Val-Loss: 1.12746
Epoch 3, Train-Loss: 1.16630,  Val-Loss: 1.01331
Epoch 4, Train-Loss: 1.06003,  Val-Loss: 0.88712
Epoch 5, Train-Loss: 0.90097,  Val-Loss: 0.78063
Epoch 6, Train-Loss: 0.79420,  Val-Loss: 0.71548
Epoch 7, Train-Loss: 0.69412,  Val-Loss: 0.70488
Epoch 8, Train-Loss: 0.67072,  Val-Loss: 0.72694
Epoch 9, Train-Loss: 0.65624,  Val-Loss: 0.75250
Epoch 10, Train-Loss: 0.71906,  Val-Loss: 0.76532
Epoch 11, Train-Loss: 0.70572,  Val-Loss: 0.76095
Epoch 12, Train-Loss: 0.70579,  Val-Loss: 0.74158
Epoch 13, Train-Loss: 0.65828,  Val-Loss: 0.71940
Epoch 14, Train-Loss: 0.63918,  Val-Loss: 0.69694
Epoch 15, Train-Loss: 0.68512,  Val-Loss: 0.68507
Epoch 16, Train-Loss: 0.62258,  Val-Loss: 0.68198
Epoch 17, Train-Loss: 0.60124,  Val-Loss: 0.68076
Epoch 18, Train-Loss: 0.62679,  Val-Loss: 0.68143
Epoch 19, Train-Loss: 0.59097,  Val-Loss: 0.68383
Epoch 20, Train-Loss: 0.59591,  Val-Loss: 0.68836
Epoch 21,

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1, Train-Loss: 0.82426,  Val-Loss: 0.72093
Epoch 2, Train-Loss: 0.82773,  Val-Loss: 0.69556
Epoch 3, Train-Loss: 0.79584,  Val-Loss: 0.68488
Epoch 4, Train-Loss: 0.74505,  Val-Loss: 0.70419
Epoch 5, Train-Loss: 0.68713,  Val-Loss: 0.73534
Epoch 6, Train-Loss: 0.72506,  Val-Loss: 0.77066
Epoch 7, Train-Loss: 0.72043,  Val-Loss: 0.77765
Epoch 8, Train-Loss: 0.68355,  Val-Loss: 0.75813
Epoch 9, Train-Loss: 0.62194,  Val-Loss: 0.72181
Epoch 10, Train-Loss: 0.60754,  Val-Loss: 0.69693
Epoch 11, Train-Loss: 0.62934,  Val-Loss: 0.68576
Epoch 12, Train-Loss: 0.65472,  Val-Loss: 0.69082
Epoch 13, Train-Loss: 0.60868,  Val-Loss: 0.69789
Epoch 14, Train-Loss: 0.59505,  Val-Loss: 0.71174
Epoch 15, Train-Loss: 0.56876,  Val-Loss: 0.72147
Epoch 16, Train-Loss: 0.58762,  Val-Loss: 0.71154
Epoch 17, Train-Loss: 0.52830,  Val-Loss: 0.67471
Epoch 18, Train-Loss: 0.47552,  Val-Loss: 0.66395
Epoch 19, Train-Loss: 0.49241,  Val-Loss: 0.71003
Epoch 20, Train-Loss: 0.44484,  Val-Loss: 0.75424
Epoch 21,

In [120]:
config.epochs = check_point
config.model_path = best_model_path

print("Best model path:", join_drive_path("log", config.model_path))
print("Model checkpoint:", config.epochs)

Best model path: /root/ADHD-EEG-ViT-myversion/log/ieee-transformer_260309004348772585_1.pt
Model checkpoint: 32


In [121]:
# clear_cache()

# trained_weights = torch.load(os.path.join(LOG_DIR, config.model_path), weights_only=True, map_location=device)

# model = ViTransformer(**model_param)
# model.load_state_dict(trained_weights)

import os
import torch

clear_cache()

# config.model_path 现在打印出来已经是一个绝对路径（Best model path 那行）
real_model_path = config.model_path
print("Loading model from:", real_model_path)
assert os.path.exists(real_model_path), f"Model file not found: {real_model_path}"

# 先加载到 CPU，避免加载阶段就占用/冲击 GPU
state_dict = torch.load(real_model_path, map_location="cpu")  # 不用 weights_only

model = ViTransformer(**model_param)
model.load_state_dict(state_dict)

# 再把模型搬到你之前选择的 device（cuda 或 cpu）
model = model.to(device)
model.eval()

Loading model from: ieee-transformer_260309004348772585_1.pt


ViTransformer(
  (proj): Conv1d(19, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (transformer): Transformer(
    (encoder): ModuleList(
      (0-3): 4 x AttentionBlock(
        (attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (feedforward): Sequential(
          (0): Linear(in_features=64, out_features=128, bias=True)
          (1): ReLU()
          (2): Linear(in_features=128, out_features=64, bias=True)
        )
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      )
    )
    (global_max_pool): Sequential(
      (0): AdaptiveMaxPool1d(output_size=1)
      (1): Dropout(p=0.1, inplace=False)
    )
    (fc): Sequential(
      (0): Flatten(start_dim=1, end_dim=-1)
      (1): Linear(in_features=64, out_features=32, bias=True)
      (2): ReLU()
      (3): Dropout(p=0.1, inplace=False)
      (4

In [122]:
#test_data_path = join_drive_path("data", data_config.test)
#test_dataset = EEGDataset(test_data_path)

test_data_path = os.path.join(DATA_DIR, data_config.test)  # test.pt
test_data = torch.load(test_data_path, weights_only=True)
test_dataset = EEGDataset(test_data)

test_dataloader = DataLoader(test_dataset, batch_size=config.batch)

metrics = evaluate(model, device, test_dataloader)

print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"F1-Score: {metrics['f1-score']:.3f}")
print(f"Recall: {metrics['recall']:.3f}")
print(f"AUC: {metrics['auc']:.3f}")

Accuracy: 0.611
F1-Score: 0.750
Recall: 1.000
AUC: 0.533


In [123]:
json_path = join_drive_path("log", f"{config.name}_{config.id}.json")
log_json(
    json_path, config=config, data=data_config, model=model_config, metrics=metrics
)

{'config': {'name': 'ieee-transformer',
  'batch': 8,
  'epochs': 32,
  'lr': 0.001,
  'enable_fp16': True,
  'grad_step': 4,
  'warmup_steps': 30,
  'lr_decay_factor': 0.5,
  'weight_decay': 0.001,
  'patience': 30,
  'id': '260309004348772585',
  'model_path': 'ieee-transformer_260309004348772585_1.pt',
  'k_folds': 5},
 'data': {'tag': 'IEEE_23',
  'train': 'ieee_train.pt',
  'test': 'ieee_test.pt',
  'val': 'ieee_val.pt',
  'channels': 19,
  'length': 9250,
  'num_classes': 2},
 'model': {'embed_dim': 64,
  'num_heads': 4,
  'num_blocks': 4,
  'block_hidden_dim': 128,
  'fc_hidden_dim': 32,
  'dropout': 0.1},
 'metrics': {'accuracy': 0.6111111111111112,
  'f1-score': 0.75,
  'recall': 1.0,
  'auc': 0.5333333333333333}}

In [124]:
# # 3.5改 检测channel-attention是否被启用取一个 batch 数据
# batch = next(iter(test_dataloader))
# print("batch type:", type(batch))
# print("batch len:", len(batch) if hasattr(batch, "__len__") else None)

# # 情况 A: (x, y) 或 [x, y]
# if isinstance(batch, (list, tuple)) and len(batch) == 2 and torch.is_tensor(batch[0]):
#     x, y = batch
#     x = x.to(device)

# # 情况 B: list[dict]
# elif isinstance(batch, list) and len(batch) > 0 and isinstance(batch[0], dict):
#     # 把 list[dict] 手动拼成 batch tensor（假设每个样本都有 "data"）
#     x = torch.stack([b["data"] for b in batch], dim=0).to(device)

# # 情况 C: dict（你原本写法适用）
# elif isinstance(batch, dict):
#     x = batch["data"].to(device)

# else:
#     raise TypeError(f"Unrecognized batch structure: {type(batch)} / example element: {type(batch[0]) if isinstance(batch, list) and batch else None}")

In [125]:
# #3.5 改
# import re
# from docx import Document

# DOCX_PATH = "/root/ADHD-EEG-ViT-myversion/data_raw/Channel_Labels.docx"

# def extract_channels_from_docx(docx_path):
#     doc = Document(docx_path)
#     tokens = []

#     # 1) 读段落
#     for p in doc.paragraphs:
#         txt = p.text.strip()
#         if txt:
#             tokens.append(txt)

#     # 2) 读表格（很多 docx 会把通道名放表格里）
#     for table in doc.tables:
#         for row in table.rows:
#             for cell in row.cells:
#                 txt = cell.text.strip()
#                 if txt:
#                     tokens.append(txt)

#     # 3) 从文本里提取像 EEG 通道名的 token
#     # 覆盖常见 10-20: Fp1, Fp2, F3, F4, C3, C4, P3, P4, O1, O2, F7, F8, T3/T4/T5/T6, T7/T8, P7/P8, Fz/Cz/Pz 等
#     pattern = re.compile(r"\b(?:Fp[12]|AF[34z]?|F[1-8z]|FC[1-6z]|C[1-6z]|CP[1-6z]|P[1-8z]|PO[1-8z]|O[12z]|T[3-8]|TP[7-8]|FT[7-8])\b", re.IGNORECASE)

#     found = []
#     for txt in tokens:
#         found.extend(pattern.findall(txt))

#     # 4) 保留原出现顺序去重 + 统一大小写格式（首字母大写，p 保持小写的通用写法）
#     def normalize(ch):
#         ch = ch.strip()
#         ch = ch[0].upper() + ch[1:]  # 粗略标准化
#         ch = ch.replace("FP", "Fp")  # 处理 Fp
#         return ch

#     ch_names = []
#     seen = set()
#     for ch in found:
#         ch = normalize(ch)
#         if ch not in seen:
#             seen.add(ch)
#             ch_names.append(ch)

#     return ch_names

# ch_names = extract_channels_from_docx(DOCX_PATH)
# print("Extracted channels:", ch_names)
# print("Count:", len(ch_names))

In [126]:
# #3.5改
# import os
# import numpy as np
# import pandas as pd
# import mne
# import matplotlib.pyplot as plt

# # 1) 取 attention 权重（长度=19）
# attn = model.last_channel_attn

# if attn is None:
#     raise RuntimeError("model.last_channel_attn is None，请先 forward 一次模型。")

# # dynamic attention: [B, 19]
# if attn.dim() == 2:
#     w = attn.mean(dim=0).detach().cpu().numpy()

# # static gating: [19]
# elif attn.dim() == 1:
#     w = attn.detach().cpu().numpy()

# else:
#     raise ValueError(f"Unexpected attention shape: {attn.shape}")

# print("Attention shape:", attn.shape)
# print("Weight shape:", w.shape)

# assert len(ch_names) == 19 and len(w) == 19, (len(ch_names), w.shape)

# # 2) 读 .ced 坐标文件（你项目根目录里那个）
# CED_PATH = "/root/ADHD-EEG-ViT-myversion/data_raw/Standard-10-20-Cap19new.ced"
# df = pd.read_csv(CED_PATH, sep="\t")

# # 你 preprocess 用过的坐标转换：[-Y, X, Z]
# ch_pos_all = {
#     row["labels"]: [-row["Y"], row["X"], row["Z"]]
#     for _, row in df[["labels", "X", "Y", "Z"]].iterrows()
# }

# # 3) 只取你这 19 个通道的位置
# missing = [ch for ch in ch_names if ch not in ch_pos_all]
# if missing:
#     raise ValueError(f"Channels not found in CED: {missing}\n"
#                      f"Tip: 可能是 T3/T4 vs T7/T8 命名差异，需要做 rename 映射。")

# pos = {ch: ch_pos_all[ch] for ch in ch_names}
# montage = mne.channels.make_dig_montage(ch_pos=pos, coord_frame="head")

# # 4) 构造 info 并画 topomap
# info = mne.create_info(ch_names=ch_names, sfreq=250, ch_types="eeg")
# info.set_montage(montage)

# fig, ax = plt.subplots(figsize=(5, 5))
# mne.viz.plot_topomap(w, info, axes=ax, contours=6, show=True)
# ax.set_title("EEG Channel Attention Topomap")
# plt.show()
# print(w)
# print("min:", w.min())
# print("max:", w.max())
# print("range:", w.max() - w.min())
# print("variance:", np.var(w))